In [1]:
import brainsss
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
%matplotlib inline
#from sklearn.cluster import AgglomerativeClustering
import scipy
import time
import h5py
import ants
import nibabel as nib
from scipy.ndimage import uniform_filter, gaussian_filter
import shutil
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.image import grid_to_graph
import gc
import sys
import warnings
from scipy.ndimage import gaussian_filter1d,gaussian_filter
from scipy.signal import butter, sosfiltfilt, filtfilt, freqz,iirnotch
import cv2
from scipy.ndimage.morphology import binary_erosion
from scipy.ndimage.morphology import binary_dilation
from scipy.ndimage import zoom
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import sklearn
import pickle
import itertools
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import pandas as pd
from skimage import measure
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.ndimage.morphology import binary_erosion, binary_dilation
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist
import csv
from functools import reduce

In [2]:
color_map_color_1=np.asarray(['#780000','#C1121F','#E0898F','#F0C4C7','#FFFFFF','#D9E6EF','#B3CDDE','#669BBC', '#003049'])[::-1]
cmap_personal = matplotlib.colors.LinearSegmentedColormap.from_list(
    'cmap', color_map_color_1)

In [3]:
later_path = '/oak/stanford/groups/trc/data/Ilana/2P/data/later/'
dataset_path = '/oak/stanford/groups/trc/data/Ilana/2P/data/'
supercluster_labels='/oak/stanford/groups/trc/data/Ilana/2P/data/later/temp_filter/clustering/supercluster_labels_total_500.npy'
fly_num=250
dff_path=os.path.join(dataset_path,f'fly_{fly_num}','dff')
timestamp_file =os.path.join(dataset_path,f'fly_{fly_num}','warp/timestamps_warp.h5')
ch_num=2
event='10flies'

In [4]:
for file in os.listdir(dff_path):
    if f'_{ch_num}_' in file:
        total_path=os.path.join(dff_path,file)

In [5]:
giant_cluster_labels = np.load(supercluster_labels)

In [6]:
%%time
with h5py.File(total_path, 'r') as hf:
    brain=hf['data']
    neural_activity = hf['data'][:].reshape(-1, np.shape(brain)[-1])
    print(np.shape(neural_activity))
    super_clust=500

    behavior_superclusters = []
    for cluster_num in range(super_clust):
        labels= giant_cluster_labels
        cluster_indicies= np.where(labels==cluster_num)[0]
        mean_signal = np.mean(neural_activity[cluster_indicies,:], axis=0)
        behavior_superclusters.append(mean_signal)
    behavior_superclusters = np.asarray(behavior_superclusters)
    del neural_activity
    gc.collect()

(4171804, 3384)
CPU times: user 8.27 s, sys: 42.5 s, total: 50.8 s
Wall time: 4min 25s


In [7]:
behavior_superclusters.shape

(500, 3384)

In [9]:
%%time
with h5py.File(timestamp_file, 'r') as hf:
    ts=hf['data']
    neural_activity = hf['data'][:].reshape(-1, np.shape(ts)[-1])
    print(np.shape(neural_activity))
    super_clust=500

    ts_superclusters = []
    for cluster_num in range(super_clust):
        labels= giant_cluster_labels
        cluster_indicies= np.where(labels==cluster_num)[0]
        print(np.max(neural_activity[cluster_indicies,:]),np.min(neural_activity[cluster_indicies,:]))
        mean_signal = np.mean(neural_activity[cluster_indicies,:], axis=0)
        ts_superclusters.append(mean_signal)
    ts_superclusters = np.asarray(ts_superclusters)
    del neural_activity
    gc.collect()

(4171804, 3384)
1800220.9 436.062
1800264.5 104.65488
1800264.5 104.65488
1800264.5 104.65488
1800264.5 104.65488
1800177.2 305.2434
1800264.5 104.65488
1800168.5 322.68588
1800264.5 104.65488
1800247.0 383.73456
1800255.8 418.6195
1800168.5 322.68588
1800159.9 279.07968
1800255.8 375.0133
1800238.4 348.8496
1800090.1 209.30975
1800081.4 218.031
1800238.4 401.17703
1800116.2 209.30975
1800063.9 235.47348
1800264.5 104.65488
1800159.9 252.91595
1800264.5 104.65488
1800194.8 357.57083
1800255.8 418.6195
1800098.8 244.19472
1800264.5 104.65488
1800194.8 348.8496
1800116.2 279.07968
1800203.5 287.8009
1800151.1 279.07968
1800264.5 104.65488
1800264.5 104.65488
1800177.2 340.12836
1800046.5 183.14603
1800220.9 348.8496
1800264.5 104.65488
1800159.9 322.68588
1800098.8 200.58852
1800264.5 104.65488
1800020.2 148.26108
1800264.5 104.65488
1800247.0 383.73456
1800264.5 104.65488
1800264.5 104.65488
1800133.6 287.8009
1800090.1 191.86728
1800194.8 331.4071
1800186.0 279.07968
1800264.5 479.6681

1800142.4 252.91595
1800098.8 296.52216
1800081.4 279.07968
1800081.4 252.91595
1800177.2 375.0133
1800151.1 305.2434
1800090.1 270.35846
1800186.0 331.4071
1800220.9 401.17703
1800186.0 383.73456
1800125.0 104.65488
1800264.5 104.65488
1800125.0 340.12836
1800116.2 270.35846
1800081.4 261.6372
1800255.8 479.66818
1800220.9 401.17703
1800264.5 104.65488
1800168.5 383.73456
1800264.5 104.65488
1800264.5 104.65488
1800159.9 366.29208
1800238.4 427.34076
1800151.1 313.96463
1800247.0 375.0133
1800203.5 357.57083
1800046.5 174.4248
1800264.5 104.65488
1800142.4 305.2434
1800186.0 366.29208
1800264.5 104.65488
1800212.1 348.8496
1800142.4 322.68588
1800264.5 488.38943
1800238.4 479.66818
1800133.6 366.29208
1800055.1 191.86728
1800142.4 252.91595
1800238.4 444.78323
1800264.5 418.6195
1800264.5 104.65488
1800238.4 427.34076
1799976.8 156.98232
1800046.5 261.6372
1800116.2 322.68588
1800125.0 209.30975
1800229.6 409.8983
1800194.8 331.4071
1800220.9 409.8983
1800264.5 104.65488
1800020.2 183

In [ ]:
ts_superclusters.shape

In [ ]:
%%time
total_path = os.path.join(later_path, '10flies_total_dict.pkl')
if os.path.exists(total_path)==False:
    print("Making total dict")
    total_data_dict = {}
    for x in sorted(os.listdir(later_dir)):
        fly_name= x[4:7]
        if fly_name in fly_num and 'fly' in x:
            print(x)
            temp_path = os.path.join(later_dir, x)
            with open(temp_path, 'rb') as file:
                dic = pickle.load(file)
                total_data_dict[fly_name] = dic
    with open(total_path, 'wb') as file:
        pickle.dump(total_data_dict, file)
else:
    print("Loading total dict")
    with open(total_path, 'rb') as file:
        total_data_dict = pickle.load(file)

In [ ]:
total_data_dict[f'{fly_num}'].keys()

In [ ]:
looms=np.asarray(total_data_dict[f'{fly_num}']['event_times'])
behavior=np.asarray(total_data_dict[f'{fly_num}']['behavior'])

In [ ]:
print(looms.shape)
print(behavior.shape)

In [ ]:
# 200 is when loom starts and 300 is when loom ends so 100=1sec
# the timestamps are in ms

In [ ]:
superclust=behavior_superclusters[0,:]
ts_sc=ts_superclusters[0,:]
first_loom=looms[0]
print(first_loom)
first_behave=behavior[0]

In [ ]:
bin1=[-600,0]
bin2=[700,1300]
loom_bins=[]
for i,loom in enumerate(looms):
    bin1_total=[loom+bin1[0],loom+bin1[1]]
    bin2_total=[loom-bin2[0],loom+bin2[1]]
    bin_edges=[bin1_total, bin2_total]
    loom_bins.append(bin_edges)
    

In [ ]:
# for loom in loom_bins:
test=loom_bins[0]
mask=(ts_superclusters>test[0][0]) & (ts_superclusters<test[0][1])
neural_data_a=np.where(mask,behavior_superclusters,np.nan)

In [ ]:
for i in range(500):
    print(np.count_nonzero(~np.isnan(neural_data_a[i,:])))